In [27]:
import torch
import numpy as np
from torch import nn as nn
import torchvision.models as models
from torch.distributions import Categorical

In [31]:
def get_model(model, in_ch=2048, out_ch=2, device="cuda"):
    '''
        path (string): model path
        in_ch (int): input channel from the penultimate convolutional feature map
        out_ch (int): the number of classes
    '''

    #model = torch.load(path, weights_only=False)

    ## From blackbox to explainable model
    fc = model.fc.state_dict()
    
    finalConv = nn.Conv2d(in_ch, out_ch, 1, 1)

    ### get the weights from the fc layer
    finalConv.load_state_dict({"weight":fc["weight"].view(out_ch, in_ch, 1, 1), "bias":fc["bias"]})
    tmp_model = nn.Sequential(*list(model.children())[:-2]+[finalConv])
    
    tmp_model = tmp_model.to(device)
    tmp_model.eval()
    return tmp_model

In [23]:
def get_tte_prediction(network, img, h=16, w=16):
    avgpool = nn.AvgPool2d(kernel_size=(h, w), stride=(1,1), padding=0) 

   
    activation = network(img)
    acti = activation.squeeze().data.cpu().numpy()
        
    out = avgpool(activation)
    prediction = out.view(out.shape[0], -1)
        
    prediction = prediction.data.cpu()

    y_prob = torch.nn.functional.softmax(prediction, dim=1)
    y_prob = np.round(y_prob.numpy(), 4)
    y_prob1 = round(y_prob[0,1].item(), 4)
    
    y_class = np.argmax(y_prob, axis = 1)

    dist = Categorical(logits=prediction)
    pred, yhat = torch.topk(dist.probs, 1)
    pred = round(pred.item(), 4)
    
    return acti, y_prob1, pred, yhat.item()

## Model

In [8]:
# Load pretrained ResNet-50
res_model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
res_model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [32]:
tte_model = get_model(res_model, in_ch=2048, out_ch=2)
tte_model

RuntimeError: shape '[2, 2048, 1, 1]' is invalid for input of size 2048000

## Inference

In [20]:
ts_img = torch.randn((1,3,512,512))
ts_img.shape

torch.Size([1, 3, 512, 512])

In [28]:
ts_img = ts_img.to('cuda')
activation, y_prob, y_class, yhat = get_tte_prediction(tte_model, ts_img)

In [29]:
activation.shape

(1000, 16, 16)